# Introducció a MapReduce




En aquest notebook veurem la tècnica MapReduce, que és una tècnica àmpliament utilitzada per a tractar grans quantitats de dades. Existeixen múltiples implementacions de MapReduce, inclòs el conegut Apatxe Hadoop. Veurem el concepte de manera intuïtiva i amb alguns exemples.

Començarem amb una tasca bàsica: donada una llista de cadenes de text, buscar com és la cadena més llarga. Això és bastant simple de fer.

In [ ]:
def find_longest_string(list_of_strings):
    longest_string = None
    longest_string_len = 0 
    for s in list_of_strings:
        if len(s) >= longest_string_len:
            longest_string_len = len(s)
            longest_string = s
    return longest_string

Per a una petita llista això funciona raonablement ràpid:

In [ ]:
list_of_strings = ['abc', 'python', 'wxyz']

%time max_length = print(find_longest_string(list_of_strings))

python
CPU times: user 1.1 ms, sys: 20 µs, total: 1.12 ms
Wall time: 1.19 ms



Però si en comptes de 3 elements tinguéssim 30.000.000?

In [ ]:
large_list_of_strings = list_of_strings*10000000
%time max_length = max(large_list_of_strings, key=len)


CPU times: user 1.38 s, sys: 2.41 ms, total: 1.39 s
Wall time: 1.38 s


El temps de resposta és ja d'alguns segons per a una operació molt simple.

Una manera de millorar el temps de càlcul és usant una CPU més potent i més ràpida. L'escalat del seu sistema mitjançant l'ús de maquinari millor i més ràpid es denomina **escalat vertical**. Però està solució no funciona sempre o no és possible.

En comptes d'aquesta opció, es podria intentar un **escalat horitzontal**, dissenyant el codi perquè pugui executar-se en paral·lel i ser més ràpid quan s'afegeixin més processadors o CPUs.


Per a fer això, cal dividir el codi en components més petits i executar càlculs en paral·lel de la següent manera:

1. dividir les nostres dades en fragments,
2. executar la funció `find_longest_string` en cada fragment en paral·lel i
3. trobar la cadena més llarga entre les sortides de tots els fragments.

El codi de la funció `find_*longest_*string` el dividirem en dos passos:

1. Calcular la longitud `len` de totes les cadenes
2. Obtenir el valor màxim `max`


In [ ]:
%%time
# paso 1:
list_of_string_lens = [len(s) for s in large_list_of_strings]
list_of_string_lens = zip(large_list_of_strings, list_of_string_lens)

# paso 2:
max_len = max(list_of_string_lens, key=lambda t: t[1])
print(max_len)

('python', 6)
CPU times: user 5.67 s, sys: 508 ms, total: 6.18 s
Wall time: 6.21 s


Ara el codi s'executa bastant més lent que abans perquè en lloc de realitzar una sola passada per totes les cadenes, fa 2: una per a calcular la longitud i una altra per a trobar el valor màxim.

Llavors el pas 2 no té com a entrada la llista original de cadenes sinó les dades preprocessaments. El pas 1 és un mapeador (**map**) perquè assigna un valor a un altre valor i el pas dos és un reductor (**reduce**) perquè obté una llista de valors i produeix un valor únic.

```
mapper = len

def reducer (p,c):
    if p[1] > c[1]:
        return p
    return c
```


Reescrivint el codi utilitzant funcions incloses en Python `map` i `redueix`, incloses en la llibreria `functools`.

In [ ]:
%%time

import functools


# paso 1
mapped = map ( len , large_list_of_strings )
mapped = zip(large_list_of_strings, mapped)

# paso 2

reduced = functools.reduce ( lambda x, y: x if x[1] > y[1] else y, mapped )

CPU times: user 5.61 s, sys: 4.38 ms, total: 5.62 s
Wall time: 5.64 s


Aquest codi fa exactament el mateix però és més genèric i, el més important, paralelitzable.

Per a paralelizar dividirem l'entrada en trossos (chunks) d'igual grandària amb la funció `chunks`.

In [ ]:
def chunks(l, n):
    n = max(1, n)
    return (l[i:i+n] for i in range(0, len(l), n))

In [ ]:
%%time
data_chunks = chunks ( large_list_of_strings , 30)

# paso 1 

reduced_all = []

for chunk in data_chunks:
  mapped_chunk = map ( len , chunk )
  mapped_chunk = zip(chunk, mapped_chunk)

  reduced_chunk = functools.reduce ( lambda x, y: x if x[1] > y[1] else y, mapped_chunk )
  reduced_all.append (reduced_chunk)

# paso 2

reduced = functools.reduce ( lambda x, y: x if x[1] > y[1] else y,reduced_all)
print (reduced)

('python', 6)
CPU times: user 7.77 s, sys: 83.8 ms, total: 7.85 s
Wall time: 7.88 s


Refeactorizando para dejar el proceso con dos funciones obtenemos:

In [ ]:
%%time
def chunks_mapper(chunk):
    mapped_chunk = map(len , chunk) 
    mapped_chunk = zip(chunk, mapped_chunk)
    return functools.reduce ( lambda x, y: x if x[1] > y[1] else y, mapped_chunk )



data_chunks = chunks ( large_list_of_strings , 30)

#paso 1:
mapped = map(chunks_mapper, data_chunks)

#paso 2
reduced = functools.reduce ( lambda x, y: x if x[1] > y[1] else y,mapped)

print (reduced)


('python', 6)
CPU times: user 6.96 s, sys: 18.6 ms, total: 6.98 s
Wall time: 7.01 s



A continuació intentarem paral·lelitzar el pas 1 usant el mòdul `multiprocesing` amb la funció `pool.map` en comptes de la funció `map` normal.

In [ ]:
%%time

import multiprocessing as mp 
pool = mp.Pool(16)


data_chunks = chunks ( large_list_of_strings , 16)

# paso 1 
mapped = pool.map(chunks_mapper, data_chunks)

# paso 2
reduced = functools.reduce ( lambda x, y: x if x[1] > y[1] else y,mapped)

print (reduced)

('python', 6)
CPU times: user 9.22 s, sys: 735 ms, total: 9.96 s
Wall time: 36.7 s



Com podeu veure la millora no és important (quan no empitjora els temps) encara que són pels problemes de paral·lelització de l'entorn i l'exclusió mútua sobre la variable `mapped`.

En qualsevol cas, s'ha arquitecturado una solució usant les funcions `map`i `redueix` que pot executar-se en paral·lel. Aquesta arquitectura té dos avantatges:



1. És escalable: si hi ha més dades es poden afegir més unitats de processament sense canviar el codi
2. És genèrica: aquesta arquitectura permet una gran quantitat de tasques reemplaçant les funcions `map` i `reduce`.

En tots dos casos se suposa que les dades són enormes i estàtiques. El que implica que dividir en fragments cada vegada és poc eficient i redundant. Pel que se suposa que les dades s'emmagatzemen en fragments (o *shards*)
d'origen.




---



Ara veurem un altre cas. Tenim una text relativament llarg que és la Declaració Universal de Drets Humans.



In [ ]:
import nltk
nltk.download('udhr')

[nltk_data] Downloading package udhr to /root/nltk_data...
[nltk_data]   Unzipping corpora/udhr.zip.


True

In [ ]:
from nltk.corpus import udhr
import re

text_udhr = udhr.raw('Spanish-Latin1')
#text_udhr = udhr.raw('Catalan-Latin1')
text_udhr[:25]

'Declaración Universal de '

Realitzant un neteja de dades i convertint-los a minúscules per a obtenir un array de cadenes

In [ ]:
%%time

def clean_word ( word ):
  return re.sub(r'[^\w\s]','',word).lower()


clean_text = clean_word ( text_udhr)
tokens = clean_text.split ()
ls = find_longest_string (tokens)

print (ls, len(ls))



arbitrariamente 15
CPU times: user 1.42 ms, sys: 23 µs, total: 1.44 ms
Wall time: 1.36 ms



I realitzat amb `map` i `reduce`.

In [ ]:
%%time

# TODO
data_chunks = chunks ( tokens , 1)

#paso 1:
mapped = map(chunks_mapper, data_chunks)

#paso 2
reduced = functools.reduce ( lambda x, y: x if x[1] > y[1] else y,mapped)

print (reduced)



('arbitrariamente', 15)
CPU times: user 48.3 ms, sys: 9.96 ms, total: 58.3 ms
Wall time: 58.7 ms



MapReduce és una tècnica essencial per a processar grans quantitats de dades i que permet una gran quantitat de tasques com comptar, buscar, aprenentatge automàtic (supervisat i no supervisat), etc.